In [1]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [2]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=7):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 7 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [3]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Qual a previsão de tempo para amanhã em Manaus?


# 3. Integração com a API do Gemini 💬

In [5]:
!pip install -q -U google-genai

In [6]:
from google import genai # Importação da biblioteca nova
from google.colab import userdata

# Configura o cliente moderno
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

gemini_response = "Ah não, Erro de integração."

try:
    # No novo SDK, a chamada é assim:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=transcription
    )

    if response.text:
        gemini_response = response.text
        print("-" * 30)
        print("✓ SUCESSO NA INTEGRAÇÃO")
        print("-" * 30)
        print(gemini_response)

except Exception as e:
    print(f"X Erro técnico: {e}")

------------------------------
✓ SUCESSO NA INTEGRAÇÃO
------------------------------
Como uma inteligência artificial, eu não tenho acesso a dados meteorológicos em tempo real para fornecer a previsão exata para "amanhã" a partir deste momento. As previsões podem mudar.

No entanto, posso te dar uma **tendência geral** para Manaus e te indicar como conseguir a previsão mais precisa:

**Tendência Geral para Manaus:**

Manaus, por estar localizada na região amazônica, geralmente apresenta um **clima equatorial, quente e úmido durante todo o ano.**

*   **Temperaturas:** É comum ter temperaturas elevadas, geralmente **acima dos 30°C**, e a sensação térmica pode ser ainda maior devido à alta umidade. As temperaturas mínimas geralmente ficam em torno dos 23-25°C.
*   **Chuva:** Pancadas de chuva, muitas vezes fortes e acompanhadas de trovoadas, são bastante frequentes, especialmente no período da **tarde ou noite**. Mesmo na estação menos chuvosa (que geralmente vai de junho a novembro), a

# 4. Sintetizando a Resposta do Gemini Como Voz (gTTS) 🔊

In [7]:
!pip install gTTS

In [8]:
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo ChatGPT e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=gemini_response, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))